객체 순환(iteration)은 파이썬의 강력한 기능 중 하나이다. 순환을 단순히 시퀀스 내부 아이템에 접근하는 방법으로 생각할 수도 있다. 

하지만 순환을 통해 할 수 있는일은 순환객체 만들기, itertools 모듈의 순환패턴 적용하기, 제너레이터(generator)함수 만들기 등 여러가지가 있다. 이번 장에서는 순환과 관련 있는 일반적인 문제를 알아본다. 

## 4.1 수동으로 이터레이터 소비

문제: 순환 가능한 아이템에 접근할 때 for 순환문을 사용하고 싶지 않다. 

해결: 수동으로 이터레이터(iterator)를 소비하려면 next()함수를 사용하고 StopIteration 예외를 처리하기 위한 코드를 직접 작성한다. 예를 들어 파일에서 줄을 읽는 코드를 보자.

In [1]:
with open('/etc/passwd') as f:
    try:
        while True:
            line = next(f)
            print(line, end= "")
    except StopIteration:
        pass

##
# User Database
# 
# Note that this file is consulted directly only when the system is running
# in single-user mode.  At other times this information is provided by
# Open Directory.
#
# See the opendirectoryd(8) man page for additional information about
# Open Directory.
##
nobody:*:-2:-2:Unprivileged User:/var/empty:/usr/bin/false
root:*:0:0:System Administrator:/var/root:/bin/sh
daemon:*:1:1:System Services:/var/root:/usr/bin/false
_uucp:*:4:4:Unix to Unix Copy Protocol:/var/spool/uucp:/usr/sbin/uucico
_taskgated:*:13:13:Task Gate Daemon:/var/empty:/usr/bin/false
_networkd:*:24:24:Network Services:/var/networkd:/usr/bin/false
_installassistant:*:25:25:Install Assistant:/var/empty:/usr/bin/false
_lp:*:26:26:Printing Services:/var/spool/cups:/usr/bin/false
_postfix:*:27:27:Postfix Mail Server:/var/spool/postfix:/usr/bin/false
_scsd:*:31:31:Service Configuration Service:/var/empty:/usr/bin/false
_ces:*:32:32:Certificate Enrollment Service:/var/empty:/usr/bin/false
_appstore:*:33:33

일반적으로 StopIteration은 순환의 끝을 알리는데 사용한다. 하지만 next()를 수동으로 사용한다면 None과 같은 종료 값을 반환하는데 사용할 수도 있다. 

토론: 대개의 경우, 순환에 for문을 사용하지만 보다 더 정교한 조절이 필요한 때도 있다. 기저에서 어떤 동작이 일어나는지 정확히 알아둘 필요가 있다. 

다음 상호 작용을 하는 예제를 통해 순환하는 동안 기본적으로 어떤일이 일어나는지 알아보자

In [2]:
items = [1, 2, 3]
it = iter(items) #items.__iter__() 실행, 이터레이터 얻기
it

In [4]:
next(it) #it.__next__() 실행, 이터레이터 실행

2

In [5]:
next(it)

3

In [6]:
next(it)

StopIteration: 

## 4.2 델리케이팅 순환

문제: 리스트, 튜플 등 순환 가능한 객체를 담은 사용자 정의 컨테이너를 만들었다. 이 컨테이너에 사용 가능한 이터레이터(iterator)를 만들고 싶다. 

해결: 일반적으로 컨테이너 순환에 사용할 __iter__()메소드만 정의해 주면 된다. 

In [7]:
class Node:
    def __init__(self, value):
        self._value = value
        self._childern = []
    def __repr__(self):
        return 'Node({!r})'.format(self._value)
    def add_child(self, node):
        self._children.append(node)
    def __iter__(self):
        return iter(self._children)